In [1]:
"""
exp12 Step 02: Continuous Replay On The Validation Span (PROTOCOL.md §5-§9)

One continuous path over the 44 validation quarters of the exp02-exp07 schedule (2015-04-17 -> 2026-04-15); the position
carries across quarters and nothing forces a buy-back. SPY bars and every VIX load are cut at 2026-04-15.
    0. REPRODUCTION (before any exp12 number): the unmodified E1 (exp04 x = 0, n = 1) and E2 (exp07 k = 4) over the 44
       periods (+108.03% with 35 exits, +115.60% with 12 exits), buy-and-hold over the full span (+235.39%) and over the
       prior-only span from 2015-07-17 (+229.74%). Any difference stops the notebook.
    1. CANDIDATE PATHS (after the fact, optimistic for the best): the 6 candidates, each replayed over the whole span; its
       return in each period minus buy-and-hold's = its period score (6 x 44 = 264 validation trials).
    2. SELECTION: mean period score over the last 4 periods; ties by the fixed rank of config.TIE_ORDER_DICT (E2M3 first).
    3. PRIOR-ONLY PATH (the honest estimate): the candidate selected at period f governs period f + 1; one continuous
       switching path with buy-and-hold, 20 random-exit and 20 random-re-entry runs, the log-excess intervals, the cost
       sensitivity, the episode scorecard (re-entry reason, VIX at exit and buy-back) and the concentration reports.
    4. INFORMATION TEST: the path must also beat the UNMODIFIED prior-only path over {E1, E2} (same selection) and the
       median of both random families.

STOP (PROTOCOL.md §8) if the prior-only path does not beat buy-and-hold: the VIX line closes definitively. A pass is only
a "candidate result pending review". RUN_MODE: "validation_only" only (the untouched window 2026-05-14 -> 2026-08-13 is
never read). Prerequisite: the raw data only. Outputs: store04_experiments/exp12_vix_fear_reentry/step02_continuous_replay_data/.
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
import time
# IMPORT PLOTLY
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp12_vix_fear_reentry import config as exp_config
# IMPORT DATA LOADING, EXECUTION, DAILY FEATURE AND VIX FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
from so.features.vix_features import get_vix_daily_feature_dict
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT THE CONTINUOUS REPLAY AND WALK-FORWARD FUNCTIONS
from so.core.continuous_replay import get_replay_period_pdf
from so.core.replay_walk_forward import run_candidate_replay_dict, run_prior_only_replay_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp12_vix_fear_reentry.rules import get_schedule_tuple, check_schedule_dict, get_rule_candidate_list, get_unmodified_candidate_list, get_rule_builder_func, \
    get_vix_scorecard_pdf, get_concentration_dict, get_yearly_log_excess_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# DEFINE THE RUN MODE: ONLY "validation_only" EXISTS FOR THIS EXPERIMENT (NO TEST WINDOW IS EVALUATED)
RUN_MODE_STR = "validation_only"
assert RUN_MODE_STR == "validation_only"
# CHECK THE REAL SCHEDULE AND THE REPRODUCTION ONLY OUTSIDE THE SYNTHETIC SMOKE RUN (scripts/smoke_notebook.py)
CHECK_REAL_DATA_BOOL = __name__ != "__smoke__"
# DEFINE THE CANDIDATE KEYS AND THE TIE-BREAKS OF THE SELECTION (PROTOCOL.md §5: THE FIXED RANK, E2M3 FIRST)
KEY_COL_STR_LIST = ["exit_rule", "modification"]
TIE_BREAK_LIST = [("tie_order", True)]
# DISPLAY THE SETTINGS
print(f"Run Mode:\t{RUN_MODE_STR}\tCandidates:\t{[c['candidate_name'] for c in get_rule_candidate_list()]}\tTie order:\t{exp_config.TIE_ORDER_DICT}\tVIX timing:\t{exp_config.VIX_VARIANT_STR}")

Run Mode:	validation_only	Candidates:	['E1M1', 'E1M2', 'E1M3', 'E2M1', 'E2M2', 'E2M3']	Tie order:	{'E2M3': 0, 'E2M1': 1, 'E2M2': 2, 'E1M3': 3, 'E1M1': 4, 'E1M2': 5}	VIX timing:	prev


In [4]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step02_continuous_replay_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "validation")

Experiment:	exp12_vix_fear_reentry	Protocol:	1.0	Config Hash:	eec3977199
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp12_vix_fear_reentry/step02_continuous_replay_data/
ℹ️ No earlier 'validation' entry of exp12_vix_fear_reentry in the trial log (config hash eec3977199).


In [5]:
# CALL FUNCTION TO READ THE MINUTE BARS AND BUILD THE SCHEDULE FROM THE FULL SESSION CALENDAR (SAME 44 QUARTERS AS exp04 STEP 02)
complete_ohlcv_pdf = get_complete_ohlcv_pdf()
fold_pdf, _ = get_schedule_tuple(sorted(complete_ohlcv_pdf["date"].unique()))
# DEFINE THE LAST VALIDATION SESSION AND CUT THE BARS THERE (NO LATER BAR IS GIVEN TO THE SIMULATOR OR THE FEATURES)
replay_end_date = pd.to_datetime(fold_pdf["valid_end"].iloc[-1]).date()
assert replay_end_date < pd.Timestamp(exp_config.UNTOUCHED_START_DATE_STR).date()
replay_ohlcv_pdf = complete_ohlcv_pdf[complete_ohlcv_pdf["date"] <= replay_end_date].reset_index(drop=True)
del complete_ohlcv_pdf
# CALL FUNCTION TO CONVERT THE CUT DATA TO ARRAYS, BUILD THE DAILY TABLE AND THE REPLAY PERIODS
ohlcv_array_dict = get_ohlcv_array_dict(replay_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
period_pdf = get_replay_period_pdf(fold_pdf, ohlcv_array_dict["session_date_list"])
# CHECK THE SCHEDULE (44 PERIODS, 2015-04-17 -> 2026-04-15, LATEST TEST START 2026-05-14 NEVER READ)
if CHECK_REAL_DATA_BOOL:
    print(f"Schedule:\t{check_schedule_dict(fold_pdf, period_pdf)}")
# CALL FUNCTION TO LOAD THE VIX AND VIX3M DAILY BARS WITH THE SAME CUTOFF AND BUILD THE _prev FEATURES
vix_cutoff_date_str = str(replay_end_date)
vix_dict = get_vix_daily_feature_dict(daily_pdf, vix_cutoff_date_str, [exp_config.VIX_VARIANT_STR])
vix_feature_pdf = vix_dict["feature_pdf"]
assert all(last_date <= replay_end_date for last_date in vix_dict["last_date_dict"].values())
# SAVE THE SCHEDULE AND THE PERIODS
write_csv_file_to_path(fold_pdf, f"{output_path_str}fold_schedule_data.csv", "W", alert_in=False)
write_csv_file_to_path(period_pdf, f"{output_path_str}replay_period_data.csv", "W", alert_in=False)
# DISPLAY THE LAST DATES LOADED
print(f"Last SPY date loaded:\t{replay_ohlcv_pdf['date'].max()}\t({len(daily_pdf):,} sessions from {replay_ohlcv_pdf['date'].min()})")
for key_str, last_date in vix_dict["last_date_dict"].items():
    print(f"Last date loaded:\t{key_str}\t{last_date}")
print(f"Folds:\t{len(fold_pdf)}\tPeriods:\t{len(period_pdf)}\t{period_pdf['period_start'].iloc[0]} -> {period_pdf['period_end'].iloc[-1]}\t({int(period_pdf['session_count'].sum()):,} sessions)")
# DISPLAY THE COVERAGE OF THE THREE FEATURES ON THE REPLAY SPAN
span_feature_pdf = vix_feature_pdf[(vix_feature_pdf["date"] >= period_pdf["period_start"].iloc[0]) & (vix_feature_pdf["date"] <= period_pdf["period_end"].iloc[-1])]
print("Missing values on the replay span:", {col: int(span_feature_pdf[col].isna().sum()) for col in [exp_config.VIX_LEVEL_COL_STR, exp_config.VIX_TERM_COL_STR, exp_config.VIX_PCT250_COL_STR]})

Schedule:	{'period_count': 44, 'first_valid_start': '2015-04-17', 'last_valid_end': '2026-04-15', 'latest_test_start': '2026-05-14'}


Last SPY date loaded:	2026-04-15	(5,353 sessions from 2005-01-03)
Last date loaded:	vix_daily	2026-04-15
Last date loaded:	vix3m_daily	2026-04-15
Folds:	44	Periods:	44	2015-04-17 -> 2026-04-15	(2,765 sessions)
Missing values on the replay span: {'vix_level_prev': 0, 'vix_term_prev': 0, 'vix_pct250_prev': 0}


In [6]:
"""
Reproduction (before any exp12 number): the unmodified exits and buy-and-hold, and the unmodified prior-only path (the
baseline of the information test, computed, not logged).
"""
# DEFINE THE RULE BUILDER (SHARED BY THE UNMODIFIED EXITS AND THE CANDIDATES)
rule_builder_func = get_rule_builder_func(daily_pdf, vix_feature_pdf)
# CALL FUNCTION TO REPLAY THE UNMODIFIED E1 AND E2 OVER THE 44 PERIODS
start_time = time.time()
unmodified_list = get_unmodified_candidate_list()
unmodified_replay_dict = run_candidate_replay_dict(ohlcv_array_dict, period_pdf, unmodified_list, rule_builder_func, exp_config.UNTOUCHED_START_DATE_STR,
                                                   exp_config.MAX_CASH_SESSIONS, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT, alert_in=False)
unmodified_summary_pdf = unmodified_replay_dict["candidate_summary_pdf"]
# CALL FUNCTION TO RUN THE UNMODIFIED PRIOR-ONLY PATH (SAME SELECTION, E2 FIRST ON TIES) WITH ITS BASELINES
unmodified_prior_dict = run_prior_only_replay_dict(ohlcv_array_dict, period_pdf, unmodified_replay_dict["candidate_period_pdf"], unmodified_list, rule_builder_func,
                                                   KEY_COL_STR_LIST, TIE_BREAK_LIST, exp_config.SELECTION_POOLED_QUARTER_COUNT, exp_config.UNTOUCHED_START_DATE_STR,
                                                   exp_config.MAX_CASH_SESSIONS, exp_config.RANDOM_RUN_COUNT, exp_config.RANDOM_EXIT_SEED_OFFSET, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
unmodified_summary_dict = unmodified_prior_dict["summary_dict"]
print(f"Unmodified exits replayed in {time.time() - start_time:.0f} s")
# COLLECT THE REPRODUCTION NUMBERS
e1_row = unmodified_summary_pdf[unmodified_summary_pdf["exit_rule"] == "E1"].iloc[0]
e2_row = unmodified_summary_pdf[unmodified_summary_pdf["exit_rule"] == "E2"].iloc[0]
reproduction_dict = {"e1_total_pct": round(100 * e1_row["strategy_total_return"], 2), "e1_exit_count": int(e1_row["exit_count"]),
                     "e2_total_pct": round(100 * e2_row["strategy_total_return"], 2), "e2_exit_count": int(e2_row["exit_count"]),
                     "buy_hold_full_pct": round(100 * unmodified_replay_dict["buy_hold_dict"]["metric_dict"]["total_return"], 2),
                     "buy_hold_prior_only_pct": round(100 * unmodified_summary_dict["buy_hold_total_return"], 2),
                     "prior_only_first_date": str(unmodified_summary_dict["first_date"])}
print(f"Reproduction:\t{reproduction_dict}")
# ASSERT THE REPRODUCTION (A DIFFERENCE STOPS THE NOTEBOOK: RECORD IT AND PARK IT)
if CHECK_REAL_DATA_BOOL:
    assert abs(reproduction_dict["e1_total_pct"] - exp_config.REPRODUCTION_E1_TOTAL_PCT) < 1e-9 and reproduction_dict["e1_exit_count"] == exp_config.REPRODUCTION_E1_EXIT_COUNT, "❌ E1 differs"
    assert abs(reproduction_dict["e2_total_pct"] - exp_config.REPRODUCTION_E2_TOTAL_PCT) < 1e-9 and reproduction_dict["e2_exit_count"] == exp_config.REPRODUCTION_E2_EXIT_COUNT, "❌ E2 differs"
    assert abs(reproduction_dict["buy_hold_full_pct"] - exp_config.REPRODUCTION_BUY_HOLD_FULL_PCT) < 1e-9, "❌ Buy-and-hold (full span) differs"
    assert abs(reproduction_dict["buy_hold_prior_only_pct"] - exp_config.REPRODUCTION_BUY_HOLD_PRIOR_ONLY_PCT) < 1e-9, "❌ Buy-and-hold (prior-only span) differs"
    assert reproduction_dict["prior_only_first_date"] == exp_config.EXPECTED_PRIOR_ONLY_START_STR, "❌ Prior-only span differs"
    print("Reproduction asserts:\tPASSED")
# DISPLAY THE UNMODIFIED PRIOR-ONLY PATH (THE BASELINE OF THE INFORMATION TEST)
print(f"Unmodified prior-only path:\t{unmodified_summary_dict['strategy_total_return']:.2%} vs buy & hold {unmodified_summary_dict['buy_hold_total_return']:.2%}"
      f"\tin market {unmodified_summary_dict['in_market_pct']:.1%}\texits {unmodified_prior_dict['simulation_dict']['metric_dict']['signal_exit_count']}"
      f"\tlog excess {unmodified_summary_dict['log_excess_annualized_log_excess']:.4f} [{unmodified_summary_dict['log_excess_ci_low']:.4f}, {unmodified_summary_dict['log_excess_ci_high']:.4f}]")
# SAVE THE BASELINE
write_csv_file_to_path(unmodified_summary_pdf, f"{output_path_str}unmodified_candidate_summary_data.csv", "W", alert_in=False)
write_csv_file_to_path(pd.DataFrame([{key: value for key, value in unmodified_summary_dict.items() if key != "baselines_beaten"}]), f"{output_path_str}unmodified_prior_only_summary_data.csv", "W", alert_in=False)
write_csv_file_to_path(unmodified_prior_dict["selection_pdf"], f"{output_path_str}unmodified_selection_data.csv", "W", alert_in=False)
# PREVIEW DATAFRAME
unmodified_summary_pdf[["candidate_name", "strategy_total_return", "buy_hold_total_return", "in_market_pct", "exit_count", "bought_back_lower_pct"]].round(4)

Unmodified exits replayed in 48 s
Reproduction:	{'e1_total_pct': np.float64(108.03), 'e1_exit_count': 35, 'e2_total_pct': np.float64(115.6), 'e2_exit_count': 12, 'buy_hold_full_pct': 235.39, 'buy_hold_prior_only_pct': 229.74, 'prior_only_first_date': '2015-07-17'}
Reproduction asserts:	PASSED
Unmodified prior-only path:	82.88% vs buy & hold 229.74%	in market 83.7%	exits 24	log excess -0.0544 [-0.0964, -0.0178]


,candidate_name,strategy_total_return,buy_hold_total_return,in_market_pct,exit_count,bought_back_lower_pct
0,E1none,1.0803,2.3539,0.8021,35,0.0857
1,E2none,1.1560,2.3539,0.8412,12,0.0833


In [7]:
"""
Candidate paths (AFTER THE FACT: the best of 6 chosen with hindsight is optimistic).
"""
# CHECK THE TRIAL BUDGET BEFORE LOGGING (EVERY STAGE OF THIS EXPERIMENT; THIS STEP ADDS 264 + 1)
candidate_list = get_rule_candidate_list()
trial_log_pdf = read_trial_log_pdf()
experiment_trial_count = int((trial_log_pdf["experiment"] == exp_config.EXPERIMENT_NAME).sum()) if len(trial_log_pdf) else 0
assert experiment_trial_count + len(candidate_list) * len(period_pdf) + 1 <= exp_config.TRIAL_BUDGET, "❌ Trial budget exceeded"
# CALL FUNCTION TO REPLAY EVERY CANDIDATE AND SCORE IT PER PERIOD
start_time = time.time()
candidate_replay_dict = run_candidate_replay_dict(ohlcv_array_dict, period_pdf, candidate_list, rule_builder_func, exp_config.UNTOUCHED_START_DATE_STR,
                                                  exp_config.MAX_CASH_SESSIONS, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
candidate_period_pdf, candidate_summary_pdf = candidate_replay_dict["candidate_period_pdf"], candidate_replay_dict["candidate_summary_pdf"]
print(f"Candidates replayed in {time.time() - start_time:.0f} s")
# LOG EVERY (CANDIDATE, PERIOD) AS A VALIDATION TRIAL (ONE WRITE, FOLD COLUMNS FILLED)
validation_trial_pdf = log_trial_pdf(exp_config, "validation", candidate_period_pdf, KEY_COL_STR_LIST, [col for col in candidate_period_pdf.columns if col.startswith("valid_")],
                                     False, fold_pdf, note_str_in="exp12 step02 candidate path, one period of the continuous replay")
print(f"Validation trials logged:\t{len(validation_trial_pdf)}\tfirst logged_ts {validation_trial_pdf['logged_ts'].iloc[0]}\ttest_window_touched: {validation_trial_pdf['test_window_touched'].unique().tolist()}")
# SAVE THE CANDIDATES
write_csv_file_to_path(candidate_period_pdf, f"{output_path_str}candidate_period_data.csv", "W", alert_in=False)
write_csv_file_to_path(candidate_summary_pdf, f"{output_path_str}candidate_summary_data.csv", "W", alert_in=False)
# DISPLAY THE OPTIMISTIC BEST CANDIDATE
best_row = candidate_summary_pdf.sort_values("strategy_total_return", ascending=False).iloc[0]
print(f"Best candidate AFTER THE FACT (optimistic):\t{best_row['candidate_name']}\t{best_row['strategy_total_return']:.2%} vs buy & hold {best_row['buy_hold_total_return']:.2%}"
      f"\tlog excess {best_row['log_excess_annualized_log_excess']:.4f} [{best_row['log_excess_ci_low']:.4f}, {best_row['log_excess_ci_high']:.4f}]\tin market {best_row['in_market_pct']:.1%}\texits {int(best_row['exit_count'])}")
# PREVIEW DATAFRAME
candidate_summary_pdf[["candidate_idx", "candidate_name", "strategy_total_return", "buy_hold_total_return", "excess_total_return", "strategy_annualized_return",
                       "buy_hold_annualized_return", "in_market_pct", "exit_count", "mean_cash_sessions", "bought_back_lower_pct", "period_windows_beating_buy_hold",
                       "log_excess_annualized_log_excess", "log_excess_ci_low", "log_excess_ci_high", "strategy_max_drawdown", "buy_hold_max_drawdown"]].round(4)

	candidate 0 {'exit_rule': 'E1', 'modification': 'M1', 'candidate_name': 'E1M1', 'tie_order': 4}: total 314.80% vs buy & hold 235.39% | in market 94.0% | exits 35


	candidate 1 {'exit_rule': 'E1', 'modification': 'M2', 'candidate_name': 'E1M2', 'tie_order': 5}: total 195.83% vs buy & hold 235.39% | in market 93.5% | exits 35


	candidate 2 {'exit_rule': 'E1', 'modification': 'M3', 'candidate_name': 'E1M3', 'tie_order': 3}: total 132.68% vs buy & hold 235.39% | in market 82.2% | exits 28


	candidate 3 {'exit_rule': 'E2', 'modification': 'M1', 'candidate_name': 'E2M1', 'tie_order': 1}: total 235.60% vs buy & hold 235.39% | in market 94.8% | exits 21


	candidate 4 {'exit_rule': 'E2', 'modification': 'M2', 'candidate_name': 'E2M2', 'tie_order': 2}: total 167.51% vs buy & hold 235.39% | in market 88.3% | exits 20


	candidate 5 {'exit_rule': 'E2', 'modification': 'M3', 'candidate_name': 'E2M3', 'tie_order': 0}: total 168.95% vs buy & hold 235.39% | in market 87.8% | exits 9
Candidates replayed in 8 s
Validation trials logged:	264	first logged_ts 2026-10-09T14:32:42	test_window_touched: [False]
Best candidate AFTER THE FACT (optimistic):	E1M1	314.80% vs buy & hold 235.39%	log excess 0.0192 [-0.0113, 0.0603]	in market 94.0%	exits 35


,candidate_idx,candidate_name,strategy_total_return,buy_hold_total_return,excess_total_return,strategy_annualized_return,buy_hold_annualized_return,in_market_pct,exit_count,mean_cash_sessions,bought_back_lower_pct,period_windows_beating_buy_hold,log_excess_annualized_log_excess,log_excess_ci_low,log_excess_ci_high,strategy_max_drawdown,buy_hold_max_drawdown
0,0,E1M1,3.1480,2.3539,0.7941,0.1384,0.1166,0.9403,35,4.7143,0.2857,19,0.0192,-0.0113,0.0603,-0.2472,-0.342
1,1,E1M2,1.9583,2.3539,-0.3956,0.1039,0.1166,0.9349,35,5.1429,0.3429,17,-0.0113,-0.0318,0.0074,-0.2430,-0.342
2,2,E1M3,1.3268,2.3539,-1.0271,0.0800,0.1166,0.8220,28,17.5714,0.1071,17,-0.0330,-0.0662,0.0016,-0.3419,-0.342
3,3,E2M1,2.3560,2.3539,0.0021,0.1167,0.1166,0.9479,21,6.8571,0.3333,19,0.0001,-0.0292,0.0315,-0.2617,-0.342
4,4,E2M2,1.6751,2.3539,-0.6788,0.0938,0.1166,0.8828,20,16.2000,0.2500,18,-0.0204,-0.0531,0.0120,-0.2034,-0.342
5,5,E2M3,1.6895,2.3539,-0.6644,0.0944,0.1166,0.8784,9,37.3333,0.1111,20,-0.0199,-0.0506,0.0060,-0.3417,-0.342


In [8]:
"""
Prior-only path (the honest validation estimate) with its baselines on the same sessions.
"""
# CALL FUNCTION TO SELECT, REPLAY THE SWITCHING PATH AND RUN THE BASELINES
start_time = time.time()
prior_dict = run_prior_only_replay_dict(ohlcv_array_dict, period_pdf, candidate_period_pdf, candidate_list, rule_builder_func, KEY_COL_STR_LIST, TIE_BREAK_LIST,
                                        exp_config.SELECTION_POOLED_QUARTER_COUNT, exp_config.UNTOUCHED_START_DATE_STR, exp_config.MAX_CASH_SESSIONS,
                                        exp_config.RANDOM_RUN_COUNT, exp_config.RANDOM_EXIT_SEED_OFFSET, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
summary_dict, baseline_summary_dict, log_excess_1m_dict = prior_dict["summary_dict"], prior_dict["baseline_summary_dict"], prior_dict["log_excess_1m_dict"]
print(f"Prior-only path and baselines in {time.time() - start_time:.0f} s")
# CHECK THAT THE PATH AND THE UNMODIFIED PATH COVER THE SAME SESSIONS AS THE REPRODUCED BUY-AND-HOLD
assert summary_dict["first_date"] == unmodified_summary_dict["first_date"] and summary_dict["last_date"] == unmodified_summary_dict["last_date"]
assert summary_dict["buy_hold_total_return"] == unmodified_summary_dict["buy_hold_total_return"]
# SAVE THE PATH
write_csv_file_to_path(prior_dict["selection_pdf"], f"{output_path_str}selection_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["baseline_pdf"], f"{output_path_str}prior_only_baseline_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["simulation_dict"]["daily_equity_pdf"], f"{output_path_str}prior_only_daily_equity_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["simulation_dict"]["transaction_pdf"], f"{output_path_str}prior_only_transaction_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["cost_sensitivity_pdf"], f"{output_path_str}prior_only_cost_sensitivity_data.csv", "W", alert_in=False)
write_csv_file_to_path(pd.DataFrame([{key: value for key, value in summary_dict.items() if key != "baselines_beaten"}]), f"{output_path_str}prior_only_summary_data.csv", "W", alert_in=False)
# DISPLAY THE PATH AGAINST BUY AND HOLD
print(f"\nPrior-only path:\t{summary_dict['first_date']} -> {summary_dict['last_date']}\t({summary_dict['session_count']:,} sessions, {summary_dict['year_count']:.2f} years)")
print(f"Total return:\tpath {summary_dict['strategy_total_return']:.2%} | buy & hold {summary_dict['buy_hold_total_return']:.2%}\t(annualized {summary_dict['strategy_annualized_return']:.2%} vs {summary_dict['buy_hold_annualized_return']:.2%})")
print(f"Time in the market:\t{summary_dict['in_market_pct']:.1%}\tExits:\t{prior_dict['simulation_dict']['metric_dict']['signal_exit_count']}\tMean cash decisions per episode:\t{prior_dict['simulation_dict']['metric_dict']['mean_cash_sessions']:.1f}")
print(f"Periods won:\t{summary_dict['period_windows_beating_buy_hold']} of {summary_dict['period_window_count']}\t(sign test p = {summary_dict['period_sign_test_p_value']:.3f})"
      f"\tmean period excess {summary_dict['period_mean_excess_return']:.2%} (SD {summary_dict['period_std_excess_return']:.2%}, t = {summary_dict['period_t_stat']:.2f})"
      f"\tMDE {summary_dict['period_mde_per_window']:.2%} per period ({summary_dict['period_mde_per_year']:.2%} per year)")
print(f"Annualized log excess:\t{summary_dict['log_excess_annualized_log_excess']:.4f}\t95% interval (6-month blocks) [{summary_dict['log_excess_ci_low']:.4f}, {summary_dict['log_excess_ci_high']:.4f}]"
      f"\t(1-month blocks [{log_excess_1m_dict['ci_low']:.4f}, {log_excess_1m_dict['ci_high']:.4f}])\tsupported: {summary_dict['log_excess_supported']}")
print(f"Sharpe:\tpath {summary_dict['strategy_sharpe_ratio']:.2f} | buy & hold {summary_dict['buy_hold_sharpe_ratio']:.2f}\tMax drawdown:\tpath {summary_dict['strategy_max_drawdown']:.2%} | buy & hold {summary_dict['buy_hold_max_drawdown']:.2%}")
print(f"PRIMARY (beats buy & hold):\t{summary_dict['primary_success']}\tComparable:\t{summary_dict['comparable_return']}\tSecondary Sharpe / drawdown:\t{summary_dict['secondary_sharpe']} / {summary_dict['secondary_drawdown']}")
print(f"Unmodified prior-only path:\t{unmodified_summary_dict['strategy_total_return']:.2%}\tpath beats it:\t{summary_dict['strategy_total_return'] > unmodified_summary_dict['strategy_total_return']}")
print(f"Random exit (median of {exp_config.RANDOM_RUN_COUNT}):\t{baseline_summary_dict['random_exit_median']:.2%}\tpath beats {baseline_summary_dict['random_exit_path_beats_share']:.0%} of runs")
print(f"Random re-entry (median of {exp_config.RANDOM_RUN_COUNT}):\t{baseline_summary_dict['random_reentry_median']:.2%}\tpath beats {baseline_summary_dict['random_reentry_path_beats_share']:.0%} of runs")
print(f"Matched probabilities:\t{prior_dict['probability_dict']}")
# APPLY THE SUCCESS AND STOPPING RULES (PROTOCOL.md §8)
path_total = summary_dict["strategy_total_return"]
information_bool = path_total > unmodified_summary_dict["strategy_total_return"] and path_total > baseline_summary_dict["random_exit_median"] and path_total > baseline_summary_dict["random_reentry_median"]
if not summary_dict["primary_success"]:
    verdict_str = "STOP"
elif information_bool:
    verdict_str = "CANDIDATE RESULT PENDING REVIEW"
else:
    verdict_str = "CANDIDATE RESULT PENDING REVIEW (information test failed: not attributable to the VIX)"
print(f"\nInformation test passed:\t{information_bool}\tVERDICT:\t{verdict_str}")
# PREVIEW THE COST SENSITIVITY
prior_dict["cost_sensitivity_pdf"].round(4)

Prior-only path and baselines in 46 s

Prior-only path:	2015-07-17 -> 2026-04-15	(2,702 sessions, 10.72 years)
Total return:	path 119.24% | buy & hold 229.74%	(annualized 7.60% vs 11.77%)
Time in the market:	89.5%	Exits:	29	Mean cash decisions per episode:	9.8
Periods won:	7 of 43	(sign test p = 1.000)	mean period excess -1.00% (SD 2.76%, t = -2.38)	MDE 1.18% per period (4.72% per year)
Annualized log excess:	-0.0377	95% interval (6-month blocks) [-0.0795, -0.0037]	(1-month blocks [-0.0780, 0.0030])	supported: False
Sharpe:	path 0.60 | buy & hold 0.71	Max drawdown:	path -27.86% | buy & hold -34.21%
PRIMARY (beats buy & hold):	False	Comparable:	False	Secondary Sharpe / drawdown:	False / False
Unmodified prior-only path:	82.88%	path beats it:	True
Random exit (median of 20):	187.43%	path beats 0% of runs
Random re-entry (median of 20):	90.60%	path beats 65% of runs
Matched probabilities:	{'exit_probability': 0.011993904513642725, 'reentry_probability': 0.10211267605633803}

Information t

,slippage_per_share,total_return,buy_hold_return,excess_return
0,0.00,1.1988,2.2975,-1.0987
1,0.01,1.1924,2.2974,-1.1050
2,0.02,1.1879,2.2973,-1.1094


In [9]:
"""
Episode scorecard of the prior-only path (every exit). S / R > 1 = bought back lower than sold; reentry_trigger = M1, M2,
original rule or end of data; vix_at_exit / vix_at_reentry = vix_level_prev at the two decisions.
"""
# BUILD AND SAVE THE SCORECARD
simulation_dict = prior_dict["simulation_dict"]
scorecard_pdf = get_vix_scorecard_pdf(prior_dict["scorecard_pdf"], simulation_dict, vix_feature_pdf)
write_csv_file_to_path(scorecard_pdf, f"{output_path_str}prior_only_scorecard_data.csv", "W", alert_in=False)
# DISPLAY THE SCORECARD
if not scorecard_pdf.empty:
    print(f"Episodes:\t{len(scorecard_pdf)}\tbought back lower:\t{int((scorecard_pdf['s_over_r'] > 1).sum())}\tproduct of S/R:\t{np.prod(scorecard_pdf['s_over_r']):.4f}"
          f"\tmean sessions out:\t{scorecard_pdf['sessions_out'].mean():.1f}\ttotal cost:\t{scorecard_pdf['cost_paid_pct'].sum():.4%} of the sale values")
    print("Re-entry reasons:", scorecard_pdf["reentry_trigger"].value_counts().to_dict())
scorecard_pdf[["exit_date", "exit_fill_price", "reentry_date", "reentry_fill_price", "s_over_r", "sessions_out", "reentry_trigger", "vix_at_exit", "vix_at_reentry",
               "buy_hold_return_out", "max_decline_out_pct", "cost_paid_pct", "open_at_end"]].round(4) if not scorecard_pdf.empty else scorecard_pdf

Episodes:	29	bought back lower:	8	product of S/R:	0.6661	mean sessions out:	9.8	total cost:	0.3137% of the sale values
Re-entry reasons: {'original rule': 19, 'M2': 6, 'M1': 4}


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,reentry_trigger,vix_at_exit,vix_at_reentry,buy_hold_return_out,max_decline_out_pct,cost_paid_pct,open_at_end
0,2015-09-22,194.15,2015-10-05,198.51,0.9780,9,original rule,20.14,20.94,0.0224,-0.0372,0.0002,False
1,2015-10-15,202.26,2015-10-23,207.59,0.9743,6,original rule,18.03,14.45,0.0263,-0.0031,0.0001,False
2,2015-11-12,204.86,2015-11-18,208.86,0.9808,4,original rule,16.06,18.84,0.0194,-0.0131,0.0001,False
3,2015-12-03,205.55,2015-12-04,209.62,0.9806,1,original rule,15.91,18.11,0.0197,-0.0006,0.0001,False
4,2015-12-09,205.37,2015-12-15,205.01,1.0018,4,M2,17.60,22.73,-0.0019,-0.0264,0.0001,False
5,2015-12-17,205.16,2015-12-29,207.51,0.9887,7,original rule,17.86,16.91,0.0114,-0.0254,0.0001,False
6,2015-12-30,205.95,2016-01-12,193.83,1.0625,8,M2,16.08,24.30,-0.0589,-0.0784,0.0002,False
7,2016-01-25,187.49,2016-03-11,202.81,0.9245,33,original rule,22.34,18.05,0.0816,-0.0342,0.0002,False
8,2016-06-27,199.56,2016-06-28,203.12,0.9825,1,original rule,25.76,23.85,0.0177,-0.0001,0.0001,False
9,2018-04-02,257.51,2018-04-03,260.63,0.9880,1,original rule,19.97,23.62,0.0120,-0.0026,0.0001,False


In [10]:
"""
Pre-registered concentration reports (no gate): (a) final equity / buy-and-hold = product of S / R, without the largest
one and two episodes; (b) the log excess per calendar year; (c) the share of the total log excess from the episodes that
overlap 2020-02-01 -> 2020-06-30.
"""
# CALCULATE THE REPORTS
path_equity_pdf, buy_hold_equity_pdf = simulation_dict["daily_equity_pdf"], prior_dict["buy_hold_dict"]["daily_equity_pdf"]
concentration_dict = get_concentration_dict(scorecard_pdf, path_equity_pdf, buy_hold_equity_pdf)
year_pdf = get_yearly_log_excess_pdf(path_equity_pdf, buy_hold_equity_pdf)
# SAVE THE REPORTS
write_csv_file_to_path(pd.DataFrame([concentration_dict]), f"{output_path_str}concentration_data.csv", "W", alert_in=False)
write_csv_file_to_path(year_pdf, f"{output_path_str}yearly_log_excess_data.csv", "W", alert_in=False)
# DISPLAY (a)
print(f"(a) final equity / buy & hold:\t{concentration_dict['equity_ratio']:.4f}\tproduct of S/R:\t{concentration_dict['product_s_over_r']:.4f}\t({concentration_dict['episode_count']} episodes)")
print(f"    without the largest log(S/R):\tequity ratio {concentration_dict['equity_ratio_without_top1']:.4f}\tproduct {concentration_dict['product_s_over_r_without_top1']:.4f}\tbeats buy & hold: {concentration_dict['beats_buy_hold_without_top1']}")
print(f"    without the two largest:\tequity ratio {concentration_dict['equity_ratio_without_top2']:.4f}\tproduct {concentration_dict['product_s_over_r_without_top2']:.4f}\tbeats buy & hold: {concentration_dict['beats_buy_hold_without_top2']}")
# DISPLAY (c)
print(f"(c) episodes overlapping {exp_config.CONCENTRATION_WINDOW_START_STR} -> {exp_config.CONCENTRATION_WINDOW_END_STR}:\t{concentration_dict['window_episode_count']}"
      f"\tsum of log(S/R) {concentration_dict['window_log_share_gain']:.4f}\ttotal log excess {concentration_dict['total_log_excess']:.4f}\tshare {concentration_dict['window_share_of_log_excess']:.2f}")
# DISPLAY (b)
print("(b) log excess per calendar year:")
year_pdf.round(4)

(a) final equity / buy & hold:	0.6649	product of S/R:	0.6661	(29 episodes)
    without the largest log(S/R):	equity ratio 0.6258	product 0.6269	beats buy & hold: False
    without the two largest:	equity ratio 0.5915	product 0.5926	beats buy & hold: False
(c) episodes overlapping 2020-02-01 -> 2020-06-30:	4	sum of log(S/R) -0.0335	total log excess -0.4081	share 0.08
(b) log excess per calendar year:


,year,session_count,strategy_log_return,buy_hold_log_return,log_excess,in_market_pct
0,2015,117,-0.1262,-0.0392,-0.0870,0.7179
1,2016,252,0.0461,0.0919,-0.0457,0.8413
2,2017,251,0.1768,0.1770,-0.0002,1.0000
3,2018,251,-0.0699,-0.0650,-0.0049,0.9124
4,2019,252,0.2017,0.2523,-0.0506,0.9762
5,2020,253,0.1159,0.1496,-0.0337,0.8656
6,2021,252,0.2389,0.2390,-0.0001,1.0000
7,2022,251,-0.2296,-0.2164,-0.0132,0.7012
8,2023,250,0.0319,0.2174,-0.1855,0.7560
9,2024,252,0.2091,0.2092,-0.0001,1.0000


In [11]:
# DISPLAY THE SELECTED CANDIDATE PER PERIOD (THE CHOICE OF PERIOD f GOVERNS PERIOD f + 1)
selection_pdf = prior_dict["selection_pdf"]
print("Selections (candidate: periods chosen):", selection_pdf.groupby("candidate_name").size().to_dict())
governed_name_list = [candidate_list[prior_dict["period_candidate_dict"][period_id]]["candidate_name"] for period_id in sorted(prior_dict["period_candidate_dict"])]
print("Periods governed by each candidate:", pd.Series(governed_name_list).value_counts().to_dict())
selection_pdf[["fold_id", "candidate_name", "pooled_score", "pooled_quarter_count"]].round(4)

Selections (candidate: periods chosen): {'E1M1': 20, 'E1M2': 3, 'E1M3': 2, 'E2M1': 7, 'E2M2': 7, 'E2M3': 5}
Periods governed by each candidate: {'E1M1': 20, 'E2M1': 7, 'E2M2': 7, 'E2M3': 5, 'E1M2': 2, 'E1M3': 2}


,fold_id,candidate_name,pooled_score,pooled_quarter_count
0,0,E2M3,0.0000,1
1,1,E1M2,0.0120,2
2,2,E2M3,0.0127,3
3,3,E1M1,0.0091,4
4,4,E1M1,0.0083,4
5,5,E2M1,0.0062,4
6,6,E2M2,-0.0027,4
7,7,E2M2,0.0000,4
8,8,E1M1,0.0000,4
9,9,E1M1,0.0001,4


In [12]:
# CREATE A FIGURE OF THE PRIOR-ONLY PATH vs THE UNMODIFIED PATH vs BUY AND HOLD (GROWTH OF 1)
fig = go.Figure()
fig.add_trace(go.Scatter(x=path_equity_pdf["date"], y=path_equity_pdf["equity"] / config.INITIAL_CAPITAL, name="prior-only path (exp12)", line={"width": 1.5}))
fig.add_trace(go.Scatter(x=unmodified_prior_dict["simulation_dict"]["daily_equity_pdf"]["date"], y=unmodified_prior_dict["simulation_dict"]["daily_equity_pdf"]["equity"] / config.INITIAL_CAPITAL,
                         name="unmodified prior-only path {E1, E2}", line={"width": 1.0}))
fig.add_trace(go.Scatter(x=buy_hold_equity_pdf["date"], y=buy_hold_equity_pdf["equity"] / config.INITIAL_CAPITAL, name="buy & hold", line={"width": 1.5}))
fig.update_layout(template="plotly_dark", width=1100, height=450, title="exp12 prior-only path vs unmodified exits vs buy-and-hold (continuous replay, growth of 1)")
fig.show()

In [13]:
# LOG ONE SUMMARY ENTRY OF THE RUN (THE CANDIDATE PERIODS ARE ALREADY LOGGED; THIS RECORDS THE HONEST ESTIMATE THAT WAS SEEN)
summary_trial_dict = log_trial_dict(exp_config, "summary", {"run_mode": RUN_MODE_STR, "selection": f"pooled {exp_config.SELECTION_POOLED_QUARTER_COUNT} periods", "path": "prior-only continuous replay"},
                                    {**{key: value for key, value in summary_dict.items() if key != "baselines_beaten"}, **baseline_summary_dict,
                                     "log_excess_1m_ci_low": log_excess_1m_dict["ci_low"], "log_excess_1m_ci_high": log_excess_1m_dict["ci_high"],
                                     "exit_count": simulation_dict["metric_dict"]["signal_exit_count"], "verdict": verdict_str, "information_test": information_bool,
                                     "unmodified_prior_only_total_return": unmodified_summary_dict["strategy_total_return"], **reproduction_dict,
                                     **{f"concentration_{key}": value for key, value in concentration_dict.items()},
                                     "best_after_the_fact_total_return": float(candidate_summary_pdf["strategy_total_return"].max())},
                                    False, note_str_in="exp12 step02 validation summary (prior-only continuous path, unmodified path, baselines, concentration)")
print(f"Summary logged:\t{summary_trial_dict['trial_id']}\ttest_window_touched: {summary_trial_dict['test_window_touched']}")

Summary logged:	20261009143330907770_eec3977199	test_window_touched: False


In [14]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,multivariate_check,1,1,0,2026-10-06T18:27:48,2026-10-06T18:27:48
1,exp01_minute_entry,signal_check,2,2,0,2026-10-05T15:17:38,2026-10-06T12:46:46
2,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
3,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
4,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
5,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
6,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
7,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
8,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
9,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
